# AI自动生成数据
1. 隐私和安全
2. 数据增强
3. 灵活
4. 成本效益
5. 监管合规
6. 模型鲁棒性和泛化能力
7. 快速原型设计
8. 控制实验
9. 真是数据不可用时的代替方案

In [24]:
import os
from sys import prefix

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model='deepseek-ai/DeepSeek-R1',
    base_url=os.environ.get('OPENAI_BASE_URL'),
    api_key=os.environ.get('OPENAI_API_KEY'),
    temperature=0.8
)

In [25]:
from langchain_experimental.synthetic_data import create_data_generation_chain

chain = create_data_generation_chain(model)

In [26]:
# 生成数据
chain.invoke(
    {
        "fields": ['蓝色', '黄色'],
        "preferences": {}
    }
)

{'fields': ['蓝色', '黄色'],
 'preferences': {},
 'text': '\n在晴朗的午后，深邃的蓝色天空中点缀着几朵白云，而明亮的黄色太阳光洒落下来，为大地披上了一层温暖而充满活力的外衣，形成了一幅宁静又生机勃勃的自然画卷。'}

In [27]:
# 生成数据
chain.invoke(
    {
        "fields": ['粑粑', '黄色'],
        "preferences": {"style": "让它像诗歌一样"}
    }
)

{'fields': ['粑粑', '黄色'],
 'preferences': {'style': '让它像诗歌一样'},
 'text': "\n根据您的字段（['粑粑', '黄色']）和偏好（{'style': '让它像诗歌一样'}），我创建了一个详细而有趣的句子，以诗歌风格呈现。句子融入了比喻和抒情元素，将“粑粑”的黄色描绘成一幅生动的自然画面，避免了粗俗感，并强调了其内在的趣味与诗意。\n\n**句子：**  \n金色的阳光洒落，那黄色的粑粑如秋叶般缓缓沉入泥土，仿佛大地低吟的隐秘诗行，唤醒沉睡的生命力。\n\n### 解释：\n- **使用所有字段**：句子中明确包含了“粑粑”和“黄色”（通过“金色的阳光”和“黄色的粑粑”体现），确保不遗漏任何元素。\n- **诗歌风格**：通过比喻（如“如秋叶般”和“大地低吟的隐秘诗行”）、节奏感（短句与长句交替）以及抒情意境，赋予句子诗意，使其更像一首微型诗歌。\n- **详细和有趣**：句子描绘了“粑粑”在自然环境中的场景（阳光、泥土），以“秋叶”的意象让黄色显得温暖而艺术，同时加入“唤醒沉睡的生命力”的隐喻，暗示其作为自然循环的一部分，增添了哲学趣味，避免直接描述带来的不适感。"}

In [28]:
from pydantic import BaseModel


# 生成一些结构化的数据： 5个步骤
# 1、定义数据模型
class MedicalBilling(BaseModel):
    patient_id: int  # 患者ID，整数类型
    patient_name: str  # 患者姓名，字符串类型
    diagnosis_code: str  # 诊断代码，字符串类型
    procedure_code: str  # 程序代码，字符串类型
    total_charge: float  # 总费用，浮点数类型
    insurance_claim_amount: float  # 保险索赔金额，浮点数类型


# 2、 提供一些样例数据，给AI
examples = [
    {
        "example": "Patient ID: 123456, Patient Name: 张娜, Diagnosis Code: J20.9, Procedure Code: 99203, Total Charge: $500, Insurance Claim Amount: $350"
    },
    {
        "example": "Patient ID: 789012, Patient Name: 王兴鹏, Diagnosis Code: M54.5, Procedure Code: 99213, Total Charge: $150, Insurance Claim Amount: $120"
    },
    {
        "example": "Patient ID: 345678, Patient Name: 刘晓辉, Diagnosis Code: E11.9, Procedure Code: 99214, Total Charge: $300, Insurance Claim Amount: $250"
    },
]

In [29]:
from langchain_core.prompts import PromptTemplate

# 3、提示模板，指导AI生成数据
openai_template = PromptTemplate(input_variables=['example'], template="{example}")

In [30]:
from langchain_experimental.tabular_synthetic_data.prompts import SYNTHETIC_FEW_SHOT_PREFIX, SYNTHETIC_FEW_SHOT_SUFFIX
from langchain_core.prompts import FewShotPromptTemplate

prompt_template = FewShotPromptTemplate(
    prefix=SYNTHETIC_FEW_SHOT_PREFIX,
    suffix=SYNTHETIC_FEW_SHOT_SUFFIX,
    examples=examples,
    example_prompt=openai_template,
    input_variables=['subject', 'extra']
)

In [31]:
prompt_template.invoke({'subject': '医疗账单', 'extra': '扩展信息'})

StringPromptValue(text='This is a test about generating synthetic data about 医疗账单. Examples below:\n\nPatient ID: 123456, Patient Name: 张娜, Diagnosis Code: J20.9, Procedure Code: 99203, Total Charge: $500, Insurance Claim Amount: $350\n\nPatient ID: 789012, Patient Name: 王兴鹏, Diagnosis Code: M54.5, Procedure Code: 99213, Total Charge: $150, Insurance Claim Amount: $120\n\nPatient ID: 345678, Patient Name: 刘晓辉, Diagnosis Code: E11.9, Procedure Code: 99214, Total Charge: $300, Insurance Claim Amount: $250\n\nNow you generate synthetic data about 医疗账单. Make sure to 扩展信息:')

In [32]:
from langchain_experimental.tabular_synthetic_data.openai import create_openai_data_generator
# 4、创建一个结构化数据的生成器
generator = create_openai_data_generator(
    output_schema=MedicalBilling,
    llm=model,
    prompt=prompt_template
)

In [33]:
# 应该只有openai模型可以用
result = generator.generate(
    subject='医疗账单',  # 信息主题
    extra='名字可以是随机的，最后使用比较生僻的人名',  # 额外的指导信息
    runs=10  # 生成10个数据
)

OutputParserException: Could not parse function call: 'function_call'
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 